Health Care Dataset Analysis

In [14]:
import pandas as pd

hdf = pd.read_csv("D:\P01_Learning Services\Prudentia\Session Materials\Datasets\healthcare_dataset.csv")
# hdf.describe()
# hdf.info()
# hdf.duplicated().count()
# hdf.head()
# hdf.tail()

for column in hdf.columns:
    print("Column Name : ",column," Null : ",hdf[column].empty)


Column Name :  Name  Null :  False
Column Name :  Age  Null :  False
Column Name :  Gender  Null :  False
Column Name :  Blood Type  Null :  False
Column Name :  Medical Condition  Null :  False
Column Name :  Date of Admission  Null :  False
Column Name :  Doctor  Null :  False
Column Name :  Hospital  Null :  False
Column Name :  Insurance Provider  Null :  False
Column Name :  Billing Amount  Null :  False
Column Name :  Room Number  Null :  False
Column Name :  Admission Type  Null :  False
Column Name :  Discharge Date  Null :  False
Column Name :  Medication  Null :  False
Column Name :  Test Results  Null :  False


Q1. Data Integrity Assessment
	• Identify inconsistent capitalization, prefixes (Dr., MD, DDS), and formatting issues.
	• Quantify cardinality before and after standardization. Use fields Patient Name, Doctor, and Hospital.
	• Determine which variables should be excluded from predictive modelling and why.

In [42]:
import pandas as pd

hdf = pd.read_csv("D:\P01_Learning Services\Prudentia\Session Materials\Datasets\healthcare_dataset.csv")

#hdf["Name"] = hdf["Name"].str.upper()#resolve capitalization for Name
#hdf["Name"]

#hdf["Name"].unique()
print(hdf["Doctor"].nunique())
print(hdf["Name"].nunique())
hdf["Name"] = hdf["Name"].str.upper()#resolve capitalization for Name
hdf["Name"] = hdf["Name"].str.replace("MR.", "", regex=True)
hdf["Name"] = hdf["Name"].str.replace("MS.", "", regex=True)
hdf["Name"] = hdf["Name"].str.replace("MRS.", "", regex=True)
hdf["Doctor"] = hdf["Doctor"].str.replace("MD", "", regex=True)
hdf["Doctor"] = hdf["Doctor"].str.replace("DDS", "", regex=True)
hdf["Doctor"] = hdf["Doctor"].str.replace("DVM", "", regex=True)
print(hdf["Doctor"].nunique())
print(hdf["Name"].nunique())

#hdf["Insurance Provider"].unique()

40341
49992
40339
40229


Q2. Missing Data & Data Consistency Audit
	• Evaluate missing values, duplicates, invalid dates, and impossible values.
	• Assess whether missingness is MCAR, MAR, or MNAR.
    • Estimate the impact of data quality issues on downstream analyses.

In [56]:
hdf.isna().sum()
hdf.duplicated().sum()

for col in hdf.columns:
    if col in hdf.columns:
        dup_count = hdf.duplicated(subset=[col]).sum()
        print(f"  {col}: {dup_count} duplicates")

date_columns = [col for col in hdf.columns if 'Date' in col or 'date' in col]
# print(f"Date columns found: {date_columns}")
# for col in date_columns:
#     try:
#         pd.to_datetime(hdf[col])
#         print(f"  {col}: Valid dates")
#     except Exception as e:
#         print(f"  {col}: ERROR - {str(e)[:80]}")

print(f"  Negative ages: {(hdf['Age'] < 0).sum()}")
print(f"  Ages > 120: {(hdf['Age'] > 120).sum()}")

  Name: 15271 duplicates
  Age: 55423 duplicates
  Gender: 55498 duplicates
  Blood Type: 55492 duplicates
  Medical Condition: 55494 duplicates
  Date of Admission: 53673 duplicates
  Doctor: 15161 duplicates
  Hospital: 15624 duplicates
  Insurance Provider: 55495 duplicates
  Billing Amount: 5500 duplicates
  Room Number: 55100 duplicates
  Admission Type: 55497 duplicates
  Discharge Date: 53644 duplicates
  Medication: 55495 duplicates
  Test Results: 55497 duplicates
  Negative ages: 0
  Ages > 120: 0


In [20]:
import pandas as pd

hdf = pd.read_csv("D:\P01_Learning Services\Prudentia\Session Materials\Datasets\healthcare_dataset.csv")

#hdf.groupby("Age")["Medical Condition"].count().sort_values(ascending=False).head(20)

df = hdf[["Name", "Medical Condition"]].drop_duplicates()
df.groupby("Medical Condition")["Name"].count()

df = hdf[["Name", "Insurance Provider"]].drop_duplicates()
df.groupby("Insurance Provider")["Name"].count().sort_values(ascending=False).head(20)

df = hdf[["Name", "Medication"]].drop_duplicates()
df.groupby("Medication")["Name"].count().sort_values(ascending=False).head(20)

df = hdf[["Name", "Admission Type"]].drop_duplicates()
df.groupby("Admission Type")["Name"].count().sort_values(ascending=False).head(20)

df = hdf[["Name", "Billing Amount"]].drop_duplicates()
df.groupby("Billing Amount")["Name"].count().sort_values(ascending=False).head(20)




Billing Amount
 51722.122739    1
 51714.300871    1
 51661.012033    1
 51634.099835    1
 51633.858435    1
 51614.052941    1
 51587.936817    1
 51567.277671    1
 51555.796105    1
 51531.964563    1
 51501.649773    1
 51452.804792    1
 51441.729053    1
 51431.977179    1
 51419.154840    1
 51415.257872    1
-599.265359      1
-600.500754      1
-614.945585      1
-652.181369      1
Name: Name, dtype: int64

Q7. Multivariate Outlier Detection
Using:
	• Age
	• Billing Amount
	• Length of Stay
Highlight records that appear normal individually but anomalous collectively.

In [ ]:
import pandas as pd

hdf = pd.read_csv("D:\P01_Learning Services\Prudentia\Session Materials\Datasets\healthcare_dataset.csv")


hdf["Date of Admission"] = pd.to_datetime(hdf["Date of Admission"], format="%d-%m-%Y")
hdf["Discharge Date"] = pd.to_datetime(hdf["Discharge Date"], format="%d-%m-%Y")

hdf["TDuration"] = (hdf["Discharge Date"] - hdf["Date of Admission"]).dt.days
# hdf["TDuration"] = hdf["TDuration"].fillna(0).astype(int)
hdf["TDuration"].sort_values(ascending=False).head(20)
hdf[["Age","Billing Amount","TDuration"]].mean()


Q1 = hdf[["Age","Billing Amount","TDuration"]].quantile(0.25)
Q3 = hdf[["Age","Billing Amount","TDuration"]].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR


# Find outliers
outliers = hdf[(hdf[["Age","Billing Amount","TDuration"]] < lower) | (hdf[["Age","Billing Amount","TDuration"]] > upper)]
outliers_count = outliers.count()
print("Outliers count:\n", outliers_count)

hdf[["Age","Billing Amount","TDuration"]].kurtosis()

# from scipy import stats 
# z_scores = stats.zscore(hdf[["Age","Billing Amount","TDuration"]]) 
# print("Z-scores:\n", z_scores)

# # Or manually: 
# mean = hdf[["Age","Billing Amount","TDuration"]].mean() 
# std = hdf[["Age","Billing Amount","TDuration"]].std() 
# z = (hdf[["Age","Billing Amount","TDuration"]] - mean) / std 
# print("Outlier ", z)
# # Find outliers (|z| > 3) 
# outliers = hdf[abs(z_scores) > 3] 
# print(len(outliers))



Outliers count:
 Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
TDuration             0
dtype: int64


Age              -1.185576
Billing Amount   -1.190630
TDuration        -1.205098
dtype: float64

Q8. Doctor/Hospital Cost Outliers
Determine whether certain doctors or hospitals generate abnormally high billing amounts.
Investigate:
	• Mean vs Median differences
	• Cost concentration
Statistical significance of deviations

In [12]:
import pandas as pd

hdf = pd.read_csv("D:\P01_Learning Services\Prudentia\Session Materials\Datasets\healthcare_dataset.csv")

hdf["Doctor"] = hdf["Doctor"].str.replace("MD", "", regex=True)
hdf["Doctor"] = hdf["Doctor"].str.replace("DDS", "", regex=True)
hdf["Doctor"] = hdf["Doctor"].str.replace("DVM", "", regex=True)
hdf["Doctor"].unique()

hdf["Hospital"] = hdf["Hospital"].str.replace("Hospital", "", regex=True)
hdf["Hospital"].unique()

#hdf.groupby(hdf["Hospital"])["Doctor"].count().sort_values(ascending=False).head(20)

hdf.groupby(hdf["Hospital"])["Doctor"].count().kurtosis()

np.float64(178.6910361269318)